# Chapter 9 — Specification and Data Issues
## University of Neuchâtel — Python for Econometrics

Topics:
1. RESET test for functional form misspecification
2. Proxy variables for unobserved regressors
3. Log transformations
4. Outlier detection and influence

**Format:** One cell per step. Manual first, statsmodels second, `np.isclose()`.

## 0 — Setup

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm
import statsmodels.stats.diagnostic as diag
from scipy import stats
import matplotlib.pyplot as plt
import wooldridge as woo
import os

In [ ]:
chapter_dir = os.path.dirname(os.path.abspath('__file__'))
os.chdir(chapter_dir)
output_dir = os.path.join(chapter_dir, 'output')
os.makedirs(output_dir, exist_ok=True)
print('Working directory:', chapter_dir)

## 1 — Load Dataset

In [ ]:
df = woo.dataWoo('wage1')
print('Shape:', df.shape)
print(df[['wage','educ','exper','tenure']].head(8))

## 2 — RESET Test (Ramsey's Regression Specification Error Test)

**Purpose:** Detect functional form misspecification (missing squared terms, wrong functional form).

**$H_0$:** The model is correctly specified.

**Procedure:**
1. Estimate baseline model, get fitted values $\hat{y}$
2. Add $\hat{y}^2$ and $\hat{y}^3$ as extra regressors
3. Test joint significance of these new terms via F-test
4. Significant → model is misspecified

### Step 2.1 — Estimate baseline model

In [ ]:
res_base = smf.ols('wage ~ educ + exper + tenure', data=df).fit()
print('Baseline model:')
print(res_base.params.round(4))
print(f'R2 = {res_base.rsquared:.4f}')

### Step 2.2 — Create yhat², yhat³ manually

In [ ]:
df['yhat']  = res_base.fittedvalues
df['yhat2'] = df['yhat']**2
df['yhat3'] = df['yhat']**3
print('Dataset after adding fitted value powers (first 5 rows):')
print(df[['wage','yhat','yhat2','yhat3']].head())

### Step 2.3 — Augmented regression

In [ ]:
res_aug = smf.ols('wage ~ educ + exper + tenure + yhat2 + yhat3', data=df).fit()
print(res_aug.summary())

### Step 2.4 — F-test on yhat2 and yhat3 manually

In [ ]:
SSR_r  = res_base.ssr
SSR_ur = res_aug.ssr
n = len(df)
k_ur = 5  # educ, exper, tenure, yhat2, yhat3
q = 2
F_reset_manual = ((SSR_r - SSR_ur) / q) / (SSR_ur / (n - k_ur - 1))
pval_reset = stats.f.sf(F_reset_manual, dfn=q, dfd=n-k_ur-1)
print(f'RESET F-stat (manual): {F_reset_manual:.4f}')
print(f'p-value:               {pval_reset:.4f}')
if pval_reset < 0.05:
    print('Reject H0: model may be misspecified')
else:
    print('Fail to reject H0: no strong evidence of misspecification')

### Step 2.5 — RESET test with statsmodels f_test

In [ ]:
f_test_reset = res_aug.f_test('yhat2=0, yhat3=0')
F_sm = f_test_reset.fvalue[0][0]
print(f'statsmodels F-stat: {F_sm:.4f}')
print(f'p-value:            {f_test_reset.pvalue:.4f}')
print(f'Match: {np.isclose(F_reset_manual, F_sm)}')

## 3 — Proxy Variables

**Problem:** We want to control for `ability` (affects wages) but can't measure it.

**Solution:** Use a **proxy** — IQ score correlates with ability.

**Dataset:** `wage2` which includes IQ scores.

### Step 3.1 — Load wage2

In [ ]:
df2 = woo.dataWoo('wage2')
print('Shape:', df2.shape)
print(df2[['lwage','educ','IQ','exper','married','south','urban','black']].head(8))

In [ ]:
print('Correlation between educ and IQ:')
print(f'  {np.corrcoef(df2["educ"].dropna(), df2["IQ"].dropna())[0,1]:.4f}')
print('(Positive → those with more education tend to have higher IQ)')

### Step 3.2 — Without proxy: lwage ~ educ (omitting ability)

In [ ]:
df2_clean = df2.dropna(subset=['lwage','educ','IQ','exper'])
res_no_proxy = smf.ols('lwage ~ educ + exper', data=df2_clean).fit()
print(f'Without IQ proxy: b_educ = {res_no_proxy.params["educ"]:.4f}')

### Step 3.3 — With proxy: add IQ

In [ ]:
res_proxy = smf.ols('lwage ~ educ + IQ + exper', data=df2_clean).fit()
print(f'With IQ proxy: b_educ = {res_proxy.params["educ"]:.4f}')
bias = res_no_proxy.params['educ'] - res_proxy.params['educ']
print(f'Ability bias estimate: {bias:.4f}')
print('Adding IQ reduced the educ coefficient → omitted variable bias confirmed')

## 4 — Log Transformations

**Why use logs?** Many economic relationships are multiplicative/percentage-based.

| Model | $\beta_1$ interpretation |
|---|---|
| level-level: $y = \beta_0 + \beta_1 x$ | $\Delta y = \beta_1 \Delta x$ (absolute) |
| log-level: $\ln y = \beta_0 + \beta_1 x$ | $\% \Delta y \approx 100\beta_1 \Delta x$ |
| level-log: $y = \beta_0 + \beta_1 \ln x$ | $\Delta y \approx (\beta_1/100)\%\Delta x$ |
| log-log: $\ln y = \beta_0 + \beta_1 \ln x$ | $\% \Delta y = \beta_1 \% \Delta x$ (elasticity) |

In [ ]:
df_log = woo.dataWoo('wage1')
df_log['lwage'] = np.log(df_log['wage'])
print('Log wage added (first 5 rows):')
print(df_log[['wage','lwage','educ']].head())

In [ ]:
res_level = smf.ols('wage ~ educ', data=df_log).fit()
res_log   = smf.ols('lwage ~ educ', data=df_log).fit()
print(f'Level-level: b_educ = {res_level.params["educ"]:.4f}  ($/hr per year of education)')
print(f'Log-level:   b_educ = {res_log.params["educ"]:.4f}  ({100*res_log.params["educ"]:.1f}% per year of education)')

In [ ]:
n_ll = len(df_log)
x_ed = df_log['educ'].values
y_lw = df_log['lwage'].values
b1_log_manual = np.sum((x_ed-x_ed.mean())*(y_lw-y_lw.mean())) / np.sum((x_ed-x_ed.mean())**2)
print(f'Manual log-level slope: {b1_log_manual:.4f}')
print(f'Statsmodels:            {res_log.params["educ"]:.4f}')
print(f'Match: {np.isclose(b1_log_manual, res_log.params["educ"])}')

## 5 — Outlier Detection

In [ ]:
res_full = smf.ols('wage ~ educ + exper + tenure', data=df_log).fit()
influence = res_full.get_influence()
stud_resid = influence.resid_studentized_external
cooks_d    = influence.cooks_distance[0]
print('Studentized residuals (first 5):', stud_resid[:5].round(3))
print(f'Observations with |stud_resid| > 3: {(np.abs(stud_resid) > 3).sum()}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12,4))
axes[0].scatter(range(len(stud_resid)), stud_resid, alpha=0.5, s=15)
axes[0].axhline(3, color='red', linestyle='--')
axes[0].axhline(-3, color='red', linestyle='--')
axes[0].set_xlabel('Observation')
axes[0].set_ylabel('Studentized Residual')
axes[0].set_title('Outlier Detection')
axes[1].scatter(range(len(cooks_d)), cooks_d, alpha=0.5, s=15)
axes[1].axhline(4/len(df_log), color='red', linestyle='--', label='4/n threshold')
axes[1].set_xlabel('Observation')
axes[1].set_ylabel("Cook's Distance")
axes[1].set_title("Cook's Distance (Influential Points)")
axes[1].legend()
plt.tight_layout()
plt.savefig(os.path.join(output_dir,'fig_ch09_01_outliers.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
sumtab = pd.DataFrame({
    'Model': ['Level-level (wage~educ)','Log-level (lwage~educ)',
              'No IQ proxy','With IQ proxy'],
    'b_educ': [
        res_level.params['educ'],
        res_log.params['educ'],
        res_no_proxy.params['educ'],
        res_proxy.params['educ']
    ],
    'R2': [
        res_level.rsquared,
        res_log.rsquared,
        res_no_proxy.rsquared,
        res_proxy.rsquared
    ]
})
print(sumtab.round(4).to_string(index=False))
sumtab.to_csv(os.path.join(output_dir,'table_ch09_specification.csv'), index=False)
print('Saved table_ch09_specification.csv')